In [4]:
import pandas as pd
import os

base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
raw_file = os.path.join(base_dir, "data", "raw", "financial_data.csv")

# Try different encodings
df = pd.read_csv(raw_file, encoding="latin1")

print("✅ Loaded successfully!")
print(df.shape)
df.head()

✅ Loaded successfully!
(9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11-08-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11-08-2016,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,06-12-2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10-11-2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10-11-2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [7]:
df = df.rename(columns={
    "Order_Date": "date",
    "Sales": "revenue",
    "Profit": "profit",
    "Category": "product",
    "Region": "region"
})
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9994 non-null   int64  
 1   Order ID       9994 non-null   object 
 2   date           9994 non-null   object 
 3   Ship Date      9994 non-null   object 
 4   Ship Mode      9994 non-null   object 
 5   Customer ID    9994 non-null   object 
 6   Customer Name  9994 non-null   object 
 7   Segment        9994 non-null   object 
 8   Country        9994 non-null   object 
 9   City           9994 non-null   object 
 10  State          9994 non-null   object 
 11  Postal Code    9994 non-null   int64  
 12  region         9994 non-null   object 
 13  Product ID     9994 non-null   object 
 14  product        9994 non-null   object 
 15  Sub-Category   9994 non-null   object 
 16  Product Name   9994 non-null   object 
 17  revenue        9994 non-null   float64
 18  Quantity

In [10]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Check how many dates failed to parse
print("Null dates:", df["date"].isnull().sum())

df["cost"] = df["revenue"] - df["profit"]
df["gross_margin"] = df["profit"] / df["revenue"]
df.head()


Null dates: 5952


,Row ID,Order ID,date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Product ID,product,Sub-Category,Product Name,revenue,Quantity,Discount,profit,cost,gross_margin
0,1,CA-2016-152156,2016-11-08,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136,220.0464,0.1600
1,2,CA-2016-152156,2016-11-08,11-11-2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820,512.3580,0.3000
2,3,CA-2016-138688,2016-06-12,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714,7.7486,0.4700
3,4,US-2015-108966,2015-10-11,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310,1340.6085,-0.4000
4,5,US-2015-108966,2015-10-11,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164,19.8516,0.1125


In [11]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["year_month"] = df["date"].dt.to_period("M")
df[["date", "year_month"]].head()

,date,year_month
0,2016-11-08,2016-11
1,2016-11-08,2016-11
2,2016-06-12,2016-06
3,2015-10-11,2015-10
4,2015-10-11,2015-10


In [12]:
monthly_df = (
    df.groupby(["year_month", "product", "region"], as_index=False)
      .agg({"revenue": "sum", "cost": "sum", "profit": "sum"})
)
monthly_df.head()

,year_month,product,region,revenue,cost,profit
0,2014-01,Furniture,Central,76.728,130.4376,-53.7096
1,2014-01,Furniture,East,9.940,6.8586,3.0814
2,2014-01,Furniture,South,2625.760,1858.0568,767.7032
3,2014-01,Office Supplies,Central,324.282,401.8054,-77.5234
4,2014-01,Office Supplies,East,19.536,14.6520,4.8840


In [ ]:
import os
os.makedirs("../data/processed", exist_ok=True)
monthly_df.to_csv("../data/processed/financial_cleaned.csv", index=False)
print("✅ Done! File saved.")